In [1]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df = data.frame
print(df)

     mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0          17.99         10.38          122.80     1001.0          0.11840   
1          20.57         17.77          132.90     1326.0          0.08474   
2          19.69         21.25          130.00     1203.0          0.10960   
3          11.42         20.38           77.58      386.1          0.14250   
4          20.29         14.34          135.10     1297.0          0.10030   
..           ...           ...             ...        ...              ...   
564        21.56         22.39          142.00     1479.0          0.11100   
565        20.13         28.25          131.20     1261.0          0.09780   
566        16.60         28.08          108.30      858.1          0.08455   
567        20.60         29.33          140.10     1265.0          0.11780   
568         7.76         24.54           47.92      181.0          0.05263   

     mean compactness  mean concavity  mean concave points  mea

Explore the dataset with head(), shape, missing values, and the target distribution.

In [12]:
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [13]:
df.shape

(569, 31)

In [14]:
df.isnull().sum()

mean radius                0
mean texture               0
mean perimeter             0
mean area                  0
mean smoothness            0
mean compactness           0
mean concavity             0
mean concave points        0
mean symmetry              0
mean fractal dimension     0
radius error               0
texture error              0
perimeter error            0
area error                 0
smoothness error           0
compactness error          0
concavity error            0
concave points error       0
symmetry error             0
fractal dimension error    0
worst radius               0
worst texture              0
worst perimeter            0
worst area                 0
worst smoothness           0
worst compactness          0
worst concavity            0
worst concave points       0
worst symmetry             0
worst fractal dimension    0
target                     0
dtype: int64

In [15]:
df["target"].value_counts()

target
1    357
0    212
Name: count, dtype: int64

Create X and y

In [16]:
X = df.drop(columns="target")
y = df["target"]

Split the data into 80% training / 20% testing using random_state=42 and stratify=y

In [18]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Create a pipeline containing:
- StandardScaler
- LogisticRegression(max_iter=5000)

In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

scaler = StandardScaler()
log_reg = LogisticRegression(max_iter=1000)

pipeline = Pipeline([
    ('scaler', scaler),
    ('log_reg', log_reg)
])



First train the pipeline using Logistic Regression's default C value

In [28]:
pipeline.fit(X_train, y_train)

,steps,"[('scaler', ...), ('log_reg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


Record its test accuracy, precision, recall, and F1 score

In [31]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, pipeline.predict(X_test))
precision = precision_score(y_test, pipeline.predict(X_test))
recall = recall_score(y_test, pipeline.predict(X_test))
f1 = f1_score(y_test, pipeline.predict(X_test))

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")      
print(f"F1 Score: {f1:.2f}")

Accuracy: 0.97
Precision: 0.97
Recall: 0.99
F1 Score: 0.98


Now use GridSearchCV with 5-fold cross-validation to test

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
C_values = [0.01, 0.1, 1, 10, 100]

param_grid = {"log_reg__C": C_values} # __ conects the pipeline step name with the parameter name

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')

grid_search.fit(X_train, y_train)

,estimator,Pipeline(step..._iter=1000))])
,param_grid,"{'log_reg__C': [0.01, 0.1, ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


Find the best C.

In [35]:
print("Best C:", grid_search.best_params_["log_reg__C"])
print("Best Cross-Validation Accuracy:", grid_search.best_score_)

Best C: 1
Best Cross-Validation Accuracy: 0.9758241758241759


Evaluate the best model on the untouched test set using the same four metrics

In [38]:
best_model = grid_search.best_estimator_

tuned_predictions = best_model.predict(X_test)

tuned_accuracy = accuracy_score(y_test, tuned_predictions)
tuned_precision = precision_score(y_test, tuned_predictions)
tuned_recall = recall_score(y_test, tuned_predictions)
tuned_f1 = f1_score(y_test, tuned_predictions)

print("Tuned Model")
print(f"Accuracy: {tuned_accuracy:.2f}")
print(f"Precision: {tuned_precision:.2f}")
print(f"Recall: {tuned_recall:.2f}")
print(f"F1 Score: {tuned_f1:.2f}")

Tuned Model
Accuracy: 0.97
Precision: 0.97
Recall: 0.99
F1 Score: 0.98


Compare the tuned model against your original model

C value =  1 is already the same thing as original regression(they will be the same most likly)

In [ ]:
print("Default Model")
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

print(f"\nBest C: {grid_search.best_params_['log_reg__C']}")

print("\nTuned Model")
print(f"Accuracy: {tuned_accuracy:.2f}")
print(f"Precision: {tuned_precision:.2f}")
print(f"Recall: {tuned_recall:.2f}")
print(f"F1 Score: {tuned_f1:.2f}")